# RouteHunter

In [1]:
import os
import pandas as pd

from routehunter import RouteHunterApp

In [2]:
app_data_dir = "rh_data"
app = RouteHunterApp.from_data_dir(app_data_dir)

[20:46:58] Invalid InChI prefix in generating InChI Key


### 1. Review

In [3]:
print(app.review())

RouteHunter: A system for the collection and distribution of reference information on chemical synthesis routes

  Search      : give a SMILES, get papers, static CASP-solved tool
                results, and predicted solvability for that molecule.
                
  Predict     : given a SMILES, get predicted solvability probability
                per CASP tool, with a link to that tool.
                
  Monitor     : browse recently published papers, ranked by predicted
                probability of containing a multi-step synthesis route
                (pre-scored offline; candidates for you to review and
                add to the CSV by hand).


RouteHunter data review:
  Targets                    : 1362
  Papers                     : 1263
  Targets with >1 paper      : 97
  Cached CASP routes         : 0
  Predicted candidate papers : 122865 (awaiting for digitalization)
  Papers by journal:
    Organic Process Research & Development   1240
    European Journal of Organic 

### 2. Search

Given a SMILES, return literature papers *and* any CASP-predicted routes cached earlier this session.

Try some molecules with positive search:  
``C#CCOC1=C(C=C(C(=C1)N2C(=O)N3CCCCC3=N2)Cl)Cl``  
``C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O``  
``C1CCC(=C(C1)CC(=O)O)N2C(=O)C=CC(=N2)C3=C4C=CC=CN4N=C3C5=CC=CC=C5``

Try some absent molecules:  
``CC(C)Cc1ccc(cc1)C(C)C(=O)O``

In [4]:
result = app.search("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.report())

Found 1 paper(s) reporting a route for this molecule:
 - [paper] Utilization of a Benzoyl Migration To Effect an Expeditious Synthesis of the Paclitaxel C-13 Side Chain (Organic Process Research & Development, 1997) doi:10.1021/op970113b

Found 2 tool(s) predicted routes for this molecule:
 - [AiZynthFinder] This molecule was solved by AiZynthFinder. See predicted routes: Cached predicted routes are not available yet.
 - [SynPlanner] This molecule was solved by SynPlanner. See predicted routes: Cached predicted routes are not available yet.


## 3. Predict

Predict a route computationally. Results are cached in memory for this session (`cache=True` by default) so a later Search this session surfaces them too — but the cache disappears when the notebook restarts; it is never written to the CSV. Uses a stub `CASPEngine` here — swap in a real open-source CASP tool (e.g. AiZynthFinder) via the same `predict_route` interface.

In [5]:
result = app.predict("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.to_dataframe().to_string())

            tool probability                                                             url
0  AiZynthFinder         84%                    https://github.com/MolecularAI/aizynthfinder
1     SynPlanner         82%  https://github.com/Laboratoire-de-Chemoinformatique/SynPlanner


In [6]:
result.to_dataframe()

,tool,probability,url
0,AiZynthFinder,84%,https://github.com/MolecularAI/aizynthfinder
1,SynPlanner,82%,https://github.com/Laboratoire-de-Chemoinforma...


### 4. Monitor

Fetch recent papers, score with a classifier, display ranked. This is **display-only** - nothing here is written into the dataset. If a candidate turns out to be a real route, the way to record it is to add a row to the CSV and reload.

In [9]:
result = app.monitor(year_min=1995, year_max=2025)
result

,journal,title,abstract,doi,publication_date,route_prob
1,Organic Process Research & Development,Practical Synthesis of a HIV Integrase Inhibitor,A practical and efficient synthesis of the pot...,10.1021/op800153y,2008-10-29,0.829994
2,Tetrahedron,An expeditious route to the synthesis of adeno...,NaN,10.1016/0040-4039(96)00632-6,1996-05-01,0.818243
3,Organic Process Research & Development,Development of a Scalable Route to the SMO Rec...,A practical and scalable route to the SMO rece...,10.1021/op300170q,2012-10-31,0.814322
6,Tetrahedron,An efficient route for synthesis of spirocycli...,NaN,10.1016/j.tetlet.2024.155250,2024-08-14,0.807527
8,Organic Process Research & Development,"Convergent, Fit-For-Purpose, Kilogram-Scale Sy...",Process research and development of a syntheti...,10.1021/op200299p,2012-01-05,0.806385
...,...,...,...,...,...,...
19495,Synlett,Extending the Utility of the Bartoli Indolizat...,A short synthesis of marinoquinolines C and E ...,10.1055/s-0032-1318137,2013-01-23,0.577577
19496,Angewandte Chemie International Edition,Catalytic Asymmetric Total Synthesis of <i>ent...,Key to success: The first catalytic asymmetric...,10.1002/anie.200906678,2010-01-08,0.577574
19497,Journal of Organic Chemistry,Modular and Stereodivergent Approach to Unbran...,An iterative strategy for the stereodivergent ...,10.1021/acs.joc.6b01051,2016-08-26,0.577573
19498,Organic Letters,Asymmetric Total Synthesis of (−)-Spirofungin ...,[chemical reaction: see text]. The stereocontr...,10.1021/ol052039k,2005-11-09,0.577572


### 5. Download

Export the dataset (or a filtered slice) as a flat table for ML training. Every row comes from the CSV — Hunter output never appears here since it's never written into the dataset.

In [10]:
config_df = pd.read_csv(os.path.join(app_data_dir, "config.csv"))
config_df

,key,path,comment
0,TargetStaticData,static/target_static_data.csv,Digitalized collection of targets
1,AizynthfinderStaticData,static/aizynthfinder_static_data.csv,AiZynthFinder solved-by-tool table
2,SynplannerStaticData,static/synplanner_static_data.csv,SynPlanner solved-by-tool table
3,MonitorStaticData,static/monitor_static_data.csv,High-confidence paper with route candidates
4,CandidateStaticData,static/candidate_static_data.csv,Medium-confidence paper with route candidates
5,AbstractTrainingData,build/abstract_training_data.csv,Training data for paper classifier
6,AizynthfinderPredictModel,model/aizynthfinder_predict_model.pickle,AiZynthFinder solvability model
7,SynplannerPredictModel,model/synplanner_predict_model.pickle,SynPlanner solvability model
8,PaperPredictModel,model/paper_predict_model.pickle,Paper-with-route classifier model
